# AI Marketing Team — A Multi-Agent Marketing Workflow

**A working multi-agent system where specialized AI agents — Researcher, Strategist, Writer, Editor, and Analyst — collaborate under a Manager/Orchestrator to plan, produce, review, and iteratively improve a marketing campaign, with a human-approval checkpoint before anything is treated as "published."**

## 1. Project Overview

Give this pipeline a one-line product description. It will:

1. **Research** the market and audience for that product.
2. **Plan** a campaign brief (target audience, funnel stage, key message, primary KPI).
3. **Write** content for three channels (social post, email, ad copy).
4. **Edit** that content against the brief — sending it back to the Writer for revision if it doesn't pass, up to a configurable retry limit.
5. Pause for **human approval** before treating content as "live."
6. Feed it (synthetic, clearly-labeled-as-fake) **performance data**, and have an **Analyst** agent extract concrete lessons.
7. Run a **second round** of content that is explicitly briefed on what round one's results implied — a genuine feedback loop, not just "generate twice."

This is not a single prompt. It's an orchestrated system with retries, structured-output parsing, a revision loop, a human-in-the-loop gate, and an audit trail — the actual shape of a production agent workflow, scaled down to run cleanly in one Colab notebook.


## 2. Real-World Marketing Use Case

This mirrors a workflow increasingly deployed inside real marketing teams and marketing-tech vendors (e.g., HubSpot's and Salesforce's emerging "agentic" campaign tools, and in-house builds at DTC brands and agencies):

- **Campaign ideation at scale.** Instead of one strategist manually researching and briefing every SKU or micro-segment, an agent team can produce a first-pass brief and content set for dozens of products/segments, with a human reviewing outputs rather than starting from a blank page.
- **Always-on optimization.** The research → content → results → re-brief loop mirrors how performance marketing teams already work (launch, measure, iterate) — the difference is the loop is faster and partially automated, with a human still gating what goes live.
- **Governance and brand safety.** The Editor agent and the human checkpoint exist because real companies cannot let an LLM post to the public unsupervised — a wrong claim, an off-brand tone, or a compliance issue is a real business risk. Building that gate *in*, rather than bolting it on later, is exactly what "responsibly deploying agent workflows" means in practice.


## 3. System Architecture

A Manager agent orchestrates five specialists in a loop with two feedback points: the **Editor → Writer** revision loop (content-level) and the **Analyst → Manager** learnings loop (campaign-level).

```
                    ┌─────────────────────┐
              ┌────▶│   Manager (Orchestrator) │◀────┐
              │     └─────────────────────┘         │
   research/  │        │        │        │           │ learnings
   strategy   │        ▼        ▼        ▼           │ feed back
   dispatch   │  Researcher  Strategist  Writer ──▶ Editor        │
              │                              ▲         │           │
              │                              └─revise──┘           │
              │                                        ▼           │
              │                              Human Approval Gate   │
              │                                        │           │
              └────────────────────────────────────────┴──▶ Analyst
```

Run the cell below for a rendered version of this diagram.


In [ ]:
# Architecture diagram — run this to visualize the workflow described above
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_architecture():
    """Renders the agent workflow as a labeled box-and-arrow diagram."""
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis("off")
    boxes = {
        "Manager\n(Orchestrator)": (0.5, 0.88),
        "Researcher": (0.12, 0.62),
        "Strategist": (0.37, 0.62),
        "Writer": (0.62, 0.62),
        "Editor": (0.87, 0.62),
        "Human\nCheckpoint": (0.5, 0.36),
        "Analyst": (0.5, 0.1),
    }
    for label, (x, y) in boxes.items():
        ax.add_patch(mpatches.FancyBboxPatch(
            (x - 0.09, y - 0.05), 0.18, 0.1,
            boxstyle="round,pad=0.01", facecolor="#EAF6F0",
            edgecolor="#1D8F63", linewidth=1.5,
        ))
        ax.text(x, y, label, ha="center", va="center", fontsize=9, color="#12161C")

    arrows = [
        ("Manager\n(Orchestrator)", "Researcher"), ("Manager\n(Orchestrator)", "Strategist"),
        ("Manager\n(Orchestrator)", "Writer"), ("Manager\n(Orchestrator)", "Editor"),
        ("Editor", "Human\nCheckpoint"), ("Human\nCheckpoint", "Analyst"),
        ("Analyst", "Manager\n(Orchestrator)"),
    ]
    for a, b in arrows:
        x1, y1 = boxes[a]
        x2, y2 = boxes[b]
        ax.annotate("", xy=(x2, y2 + 0.055), xytext=(x1, y1 - 0.055),
                    arrowprops=dict(arrowstyle="->", color="#2C6DA0", lw=1.3))

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title("Agent Workflow Architecture", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

plot_architecture()


## 4. Required APIs and Data Sources

| Source | Used for | Required? |
|---|---|---|
| **Anthropic API** (Claude) | Powers every agent's reasoning and text generation | Optional — notebook runs in **MOCK_MODE** without it |
| Synthetic performance generator (built into this notebook) | Stands in for a real ads platform's reporting API | Always available, no key needed |

**Why MOCK_MODE exists:** this notebook is designed to run — and demonstrate the full architecture — with zero API keys and zero cost, which matters for anyone reviewing it on GitHub. Add a real `ANTHROPIC_API_KEY` and every agent switches to genuine LLM calls with no other code changes.

**Upgrade path for real deployment:** swap the synthetic performance generator for a real connector (Meta Marketing API, Google Ads API, or your analytics warehouse) and give the Researcher agent a live web-search tool (e.g., Tavily) instead of relying on the model's own knowledge. Both are noted again in Section 15.

## 5. Required Python Libraries

```
anthropic       # Claude API client (only required outside MOCK_MODE)
matplotlib      # charts
pandas          # audit-trail / metrics tables
numpy           # aggregate stats
dataclasses     # structured records (standard library, Python 3.7+)
```


In [ ]:
# Install/verify dependencies (safe to re-run)
!pip install anthropic matplotlib pandas numpy -q
print("Dependencies ready.")


### Configuration

Reads `ANTHROPIC_API_KEY` from Colab's secret manager (recommended — click the 🔑 icon in the left sidebar) or an environment variable. If neither is found, the notebook runs fully in **MOCK_MODE** — every agent still runs, using realistic canned responses, so the whole pipeline is demonstrable with no cost and no key.


In [ ]:
import os
import re
import json
import time
import random
from dataclasses import dataclass
from typing import List, Dict, Optional
from datetime import datetime, timezone

MODEL = "claude-sonnet-5"   # swap for any current Claude model string
MOCK_MODE = False
ANTHROPIC_API_KEY = None

# 1) Try Colab's secret manager
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass

# 2) Fall back to a plain environment variable (e.g. for local Jupyter)
if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

try:
    import anthropic
except ImportError:
    anthropic = None
    print("`anthropic` package not installed — forcing MOCK_MODE.")
    MOCK_MODE = True

if not ANTHROPIC_API_KEY:
    print("No ANTHROPIC_API_KEY found — running in MOCK_MODE "
          "(the full pipeline still runs end to end, using canned responses; no cost, no key needed).")
    MOCK_MODE = True
elif anthropic is not None:
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    print("Live API key detected — agents will call Claude for real.")


## 6. Folder/File Structure

This notebook is intentionally single-file for Colab. If you promote it to a GitHub repo (recommended for the portfolio version), split it like this:

```
ai-marketing-team/
├── README.md                  # project overview + demo GIF/screenshot
├── requirements.txt           # anthropic, matplotlib, pandas, numpy
├── .env.example                # ANTHROPIC_API_KEY=
├── src/
│   ├── config.py               # MODEL, MOCK_MODE, client setup
│   ├── schemas.py               # CampaignBrief, ContentPiece, PerformanceRecord
│   ├── agents/
│   │   ├── base.py              # BaseAgent
│   │   ├── researcher.py
│   │   ├── strategist.py
│   │   ├── writer.py
│   │   ├── editor.py
│   │   └── analyst.py
│   ├── orchestrator.py          # MarketingTeamOrchestrator
│   ├── performance.py           # synthetic data generator / real API connector
│   └── visualizations.py
├── notebooks/
│   └── demo.ipynb                # this notebook, trimmed to a walkthrough
└── tests/
    └── test_agents.py            # unit tests with mocked LLM responses
```


## 7. Step-by-Step Build Guide

1. **Config & schemas first.** Get `MOCK_MODE` switching cleanly before writing a single agent — it's what makes every later step testable without spending API credits.
2. **One agent, fully working** (Researcher). Confirm the retry/error-handling pattern before copying it into the rest.
3. **Add Strategist**, including the structured-JSON-output parsing with a safe fallback — this is the trickiest reliability point in the whole system.
4. **Add Writer + Editor together**, since they only make sense as a pair (the revision loop).
5. **Add the synthetic performance generator and Analyst.**
6. **Write the Orchestrator last** — it's just wiring together pieces you've already tested individually.
7. **Add visualizations and the metrics summary** once you have a full run's output to plot.


## 8. Data Collection Pipeline

There's no external dataset here — the "data" this pipeline collects is what each agent produces as it runs: research notes, a structured campaign brief, content drafts, editor verdicts, and (synthetic) performance numbers. Defining clean, typed records for each of these up front is what lets every later agent consume the previous agent's output reliably.


In [ ]:
@dataclass
class CampaignBrief:
    """The strategist's output — the single source of truth every downstream agent reads from."""
    product: str
    target_audience: str = ""
    funnel_stage: str = ""      # "awareness" | "consideration" | "conversion"
    key_message: str = ""
    primary_kpi: str = ""


@dataclass
class ContentPiece:
    """One piece of marketing content moving through the writer -> editor loop."""
    channel: str                # "social_post" | "email" | "ad_copy"
    text: str
    status: str = "draft"       # draft | needs_revision | approved | approved_with_flag
    revision_notes: str = ""
    revision_loops: int = 0


@dataclass
class PerformanceRecord:
    """Synthetic (clearly-fake) performance numbers standing in for a real ads-platform report."""
    channel: str
    ctr: float
    conversion_rate: float
    cac: float
    roas: float


## 9. Data Cleaning & Feature Engineering

The equivalent step here isn't cleaning a CSV — it's turning an LLM's free-text response into a clean, validated structured record. This is the single most failure-prone part of any real agent system (models occasionally wrap JSON in prose, or produce near-JSON), so it gets one shared, defensively-written utility function used by every agent that needs structured output, with an explicit, logged fallback rather than a silent crash.


In [ ]:
def safe_json_extract(raw_text: str, fallback: dict) -> dict:
    """
    Pulls the first {...} JSON object out of a raw LLM response and parses it.
    Falls back to a safe default dict (rather than raising) if parsing fails,
    since a malformed structured-output response should degrade gracefully,
    not crash a multi-agent pipeline mid-run.
    """
    try:
        match = re.search(r"\{.*\}", raw_text, re.DOTALL)
        payload = match.group(0) if match else raw_text
        return json.loads(payload)
    except (json.JSONDecodeError, AttributeError) as e:
        print(f"[safe_json_extract] Falling back to defaults — could not parse: {e}")
        return fallback


In [ ]:
# Quick sanity check of the extractor against a messy, realistic LLM response
_test_raw = 'Sure, here you go:\n{"approved": false, "notes": "too aggressive"}\nLet me know if you need changes.'
assert safe_json_extract(_test_raw, {}) == {"approved": False, "notes": "too aggressive"}
assert safe_json_extract("not json at all", {"approved": True}) == {"approved": True}
print("safe_json_extract: sanity checks passed.")


## 10. Core Models/Algorithms — The Agents

Every agent shares one base class that handles calling the LLM with exponential-backoff retries and a mock-mode fallback, so each specialist agent only has to define its own prompt and how it turns a response into structured output.


In [ ]:
class BaseAgent:
    """
    Shared LLM-calling logic for every agent: retries with exponential backoff
    on transient API errors, and a clean fallback to a mock response when
    MOCK_MODE is on (or no API key is configured).
    """

    def __init__(self, name: str, system_prompt: str, max_retries: int = 3):
        self.name = name
        self.system_prompt = system_prompt
        self.max_retries = max_retries

    def _call_llm(self, user_message: str) -> str:
        if MOCK_MODE:
            return self._mock_response(user_message)

        last_error = None
        for attempt in range(1, self.max_retries + 1):
            try:
                response = client.messages.create(
                    model=MODEL,
                    max_tokens=1024,
                    system=self.system_prompt,
                    messages=[{"role": "user", "content": user_message}],
                )
                return response.content[0].text
            except Exception as e:
                last_error = e
                wait = 2 ** attempt
                print(f"[{self.name}] API error (attempt {attempt}/{self.max_retries}): {e}. "
                      f"Retrying in {wait}s...")
                time.sleep(wait)

        # Every retry failed — fail loudly rather than silently returning bad data.
        raise RuntimeError(f"{self.name} failed after {self.max_retries} attempts: {last_error}")

    def _mock_response(self, user_message: str) -> str:
        return "[MOCK OUTPUT]"

    def run(self, *args, **kwargs):
        raise NotImplementedError("Each agent must implement its own run().")


In [ ]:
class ResearcherAgent(BaseAgent):
    """Looks up market/competitor context for the given product."""

    def __init__(self):
        super().__init__(
            name="Researcher",
            system_prompt=(
                "You are a market research analyst. Given a product description, "
                "produce a concise market/competitor summary: target market size, "
                "2-3 likely competitors and their positioning, and 2-3 audience pain points. "
                "Be specific and concise, under 200 words."
            ),
        )

    def run(self, product_description: str) -> str:
        prompt = f"Product: {product_description}\n\nProvide the market research summary."
        return self._call_llm(prompt)

    def _mock_response(self, user_message: str) -> str:
        return (
            "Market summary (mock): Growing niche market with 3 established competitors "
            "focused on convenience and taste. Audience pain points: price sensitivity, "
            "skepticism about health claims, decision fatigue from too many similar options."
        )


In [ ]:
class StrategistAgent(BaseAgent):
    """Turns product + research into a structured CampaignBrief."""

    def __init__(self):
        super().__init__(
            name="Strategist",
            system_prompt=(
                "You are a marketing strategist. Given a product and market research, "
                "produce a campaign brief as STRICT JSON with keys: "
                "target_audience, funnel_stage (one of: awareness, consideration, conversion), "
                "key_message, primary_kpi. No prose, JSON only."
            ),
        )

    def run(self, product_description: str, research_summary: str) -> CampaignBrief:
        prompt = (
            f"Product: {product_description}\nResearch: {research_summary}\n\n"
            "Return the campaign brief JSON."
        )
        raw = self._call_llm(prompt)
        fallback = {
            "target_audience": "general audience",
            "funnel_stage": "awareness",
            "key_message": f"Discover {product_description}.",
            "primary_kpi": "CTR",
        }
        data = safe_json_extract(raw, fallback)
        return CampaignBrief(
            product=product_description,
            target_audience=data.get("target_audience", fallback["target_audience"]),
            funnel_stage=data.get("funnel_stage", fallback["funnel_stage"]),
            key_message=data.get("key_message", fallback["key_message"]),
            primary_kpi=data.get("primary_kpi", fallback["primary_kpi"]),
        )

    def _mock_response(self, user_message: str) -> str:
        return json.dumps({
            "target_audience": "Health-conscious gym-goers aged 22-35",
            "funnel_stage": "consideration",
            "key_message": "Real protein, real taste, no compromises.",
            "primary_kpi": "conversion_rate",
        })


In [ ]:
class WriterAgent(BaseAgent):
    """Drafts marketing copy for a given channel, brief, and (optionally) editor feedback."""

    CHANNELS = ["social_post", "email", "ad_copy"]

    def __init__(self):
        super().__init__(
            name="Writer",
            system_prompt=(
                "You are a marketing copywriter. Write on-brand marketing content "
                "for the given channel, audience and message. Keep it tight and punchy."
            ),
        )
        self._current_channel = "ad_copy"   # used only by the mock fallback

    def run(self, brief: CampaignBrief, channel: str, revision_notes: str = "") -> ContentPiece:
        self._current_channel = channel
        prompt = (
            f"Channel: {channel}\n"
            f"Target audience: {brief.target_audience}\n"
            f"Funnel stage: {brief.funnel_stage}\n"
            f"Key message: {brief.key_message}\n"
        )
        if revision_notes:
            prompt += f"\nEditor feedback to address: {revision_notes}\n"
        prompt += "\nWrite the content now."
        text = self._call_llm(prompt)
        return ContentPiece(channel=channel, text=text)

    def _mock_response(self, user_message: str) -> str:
        samples = {
            "social_post": "Real protein. Real taste. Zero compromises. Try it today. #FuelYourGains",
            "email": ("Subject: Your gains just got tastier\n\n"
                      "Meet the protein bar that finally doesn't taste like cardboard."),
            "ad_copy": "Stop settling for chalky protein bars. 20g of plant protein that actually tastes good.",
        }
        return samples.get(self._current_channel, "[MOCK CONTENT]")


In [ ]:
class EditorAgent(BaseAgent):
    """Reviews a ContentPiece against the brief; approves or sends it back with notes."""

    def __init__(self):
        super().__init__(
            name="Editor",
            system_prompt=(
                'You are a brand editor. Review the content against the campaign brief. '
                'Respond with STRICT JSON: {"approved": true/false, "notes": "..."}. '
                "Reject if off-brand, off-message, or factually risky. Be concise."
            ),
        )

    def run(self, content: ContentPiece, brief: CampaignBrief) -> ContentPiece:
        prompt = (
            f"Campaign key message: {brief.key_message}\n"
            f"Audience: {brief.target_audience}\n"
            f"Content ({content.channel}):\n{content.text}\n\n"
            "Review and return the JSON verdict."
        )
        raw = self._call_llm(prompt)
        data = safe_json_extract(raw, {"approved": True, "notes": ""})
        content.status = "approved" if data.get("approved", True) else "needs_revision"
        content.revision_notes = data.get("notes", "")
        return content

    def _mock_response(self, user_message: str) -> str:
        # Approves ~70% of the time so the revision loop actually gets demonstrated on a typical run.
        approved = random.random() > 0.3
        notes = "" if approved else "Tone is slightly too aggressive for the target audience — soften the CTA."
        return json.dumps({"approved": approved, "notes": notes})


In [ ]:
class AnalystAgent(BaseAgent):
    """Reads performance data and produces concrete recommendations for the next round."""

    def __init__(self):
        super().__init__(
            name="Analyst",
            system_prompt=(
                "You are a performance marketing analyst. Given performance metrics "
                "per channel, identify what worked, what didn't, and give 2-3 concrete "
                "recommendations for the next round of content. Be specific and concise."
            ),
        )

    def run(self, performance: List[PerformanceRecord]) -> str:
        lines = [
            f"{p.channel}: CTR={p.ctr:.2%}, Conversion={p.conversion_rate:.2%}, "
            f"CAC=${p.cac:.2f}, ROAS={p.roas:.2f}x"
            for p in performance
        ]
        prompt = "Performance data:\n" + "\n".join(lines) + "\n\nWhat should round 2 do differently?"
        return self._call_llm(prompt)

    def _mock_response(self, user_message: str) -> str:
        return (
            "Recommendations (mock): The email channel underperformed on conversion — "
            "sharpen the subject line and lead with the strongest benefit. "
            "Social had the best CTR — double down on that tone and format for ad copy too."
        )


In [ ]:
def generate_synthetic_performance(
    content_pieces: List[ContentPiece], seed: Optional[int] = None
) -> List[PerformanceRecord]:
    """
    Produces believable, clearly-synthetic performance numbers per content piece,
    standing in for a real ads-platform report (see Section 4 for the upgrade path
    to a live connector). Channel-specific baselines keep the numbers plausible
    (email typically has much higher CTR than paid social/display, for instance).
    """
    rng = random.Random(seed)
    baseline = {"social_post": (0.02, 0.08), "email": (0.15, 0.25), "ad_copy": (0.01, 0.04)}
    records = []
    for piece in content_pieces:
        ctr_range = baseline.get(piece.channel, (0.01, 0.05))
        ctr = rng.uniform(*ctr_range)
        conversion_rate = ctr * rng.uniform(0.05, 0.2)
        cac = rng.uniform(8, 45)
        roas = rng.uniform(0.5, 4.5)
        records.append(PerformanceRecord(
            channel=piece.channel, ctr=ctr, conversion_rate=conversion_rate, cac=cac, roas=roas
        ))
    return records


def human_checkpoint(content_pieces: List[ContentPiece], auto_approve: bool = False) -> bool:
    """
    The governance gate: nothing is treated as 'published' without explicit approval.
    auto_approve=True lets the notebook run start-to-finish non-interactively
    (e.g. via 'Run All'); set it to False to actually pause for real input().
    """
    print("\n--- HUMAN APPROVAL CHECKPOINT ---")
    for piece in content_pieces:
        print(f"\n[{piece.channel.upper()}] ({piece.status})\n{piece.text}\n")

    if auto_approve:
        print("Auto-approve mode ON — treating all content as approved.")
        return True

    answer = input("Approve this content for 'publishing'? (y/n): ").strip().lower()
    return answer == "y"


### The Manager / Orchestrator

This is the part that actually makes the system "agentic" rather than a fixed script: it decides whether a piece of content needs another revision pass, when to stop retrying, and how to brief round two based on what the Analyst found — not a hardcoded sequence.


In [ ]:
class MarketingTeamOrchestrator:
    """
    Coordinates the full workflow: research -> strategy -> (write <-> edit)* -> human
    approval -> synthetic performance -> analysis -> a second, learnings-informed round.
    Keeps a full audit trail of every step for the metrics/export sections below.
    """

    def __init__(self, max_revision_loops: int = 2):
        self.researcher = ResearcherAgent()
        self.strategist = StrategistAgent()
        self.writer = WriterAgent()
        self.editor = EditorAgent()
        self.analyst = AnalystAgent()
        self.max_revision_loops = max_revision_loops
        self.history: List[dict] = []

    def _log(self, step: str, detail: str):
        self.history.append({
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "step": step,
            "detail": detail,
        })

    def produce_content_with_review(self, brief: CampaignBrief, channel: str) -> ContentPiece:
        """The Writer <-> Editor loop for a single channel, capped at max_revision_loops."""
        revision_notes = ""
        content = None
        for loop in range(self.max_revision_loops + 1):
            content = self.writer.run(brief, channel, revision_notes)
            content = self.editor.run(content, brief)
            self._log("editor_review", f"{channel} loop {loop}: {content.status}")

            if content.status == "approved":
                content.revision_loops = loop
                return content

            revision_notes = content.revision_notes
            print(f"[Manager] {channel} sent back for revision (loop {loop + 1}): {revision_notes}")

        # Retries exhausted — publish the best-effort version but flag it rather than
        # silently treating it as fully approved.
        print(f"[Manager] Max revision loops reached for {channel}; flagging for manual review.")
        content.status = "approved_with_flag"
        content.revision_loops = self.max_revision_loops
        return content

    def run_round(
        self, product_description: str,
        brief: Optional[CampaignBrief] = None, learnings: str = "",
    ) -> Dict:
        if brief is None:
            print("[Manager] Kicking off Researcher...")
            research = self.researcher.run(product_description)
            self._log("research", research)

            print("[Manager] Kicking off Strategist...")
            brief = self.strategist.run(product_description, research)
            self._log("strategy", str(brief))
        else:
            print("[Manager] Round 2 — briefing with prior learnings...")
            brief.key_message = f"{brief.key_message} (Refined based on results: {learnings[:100]}...)"

        content_pieces = []
        for channel in WriterAgent.CHANNELS:
            print(f"[Manager] Producing + reviewing content for: {channel}")
            content_pieces.append(self.produce_content_with_review(brief, channel))

        return {"brief": brief, "content": content_pieces}

    def run_full_campaign(self, product_description: str, auto_approve_human: bool = True) -> Dict:
        print("=" * 60); print("ROUND 1"); print("=" * 60)
        round1 = self.run_round(product_description)

        approved = human_checkpoint(round1["content"], auto_approve=auto_approve_human)
        if not approved:
            print("[Manager] Human rejected round 1 content — stopping here.")
            return {"round1": round1}

        performance = generate_synthetic_performance(round1["content"], seed=42)
        self._log("performance_r1", str(performance))

        print("\n[Manager] Kicking off Analyst on round 1 performance...")
        learnings = self.analyst.run(performance)
        self._log("analysis", learnings)
        print(f"[Analyst] {learnings}")

        print("\n" + "=" * 60); print("ROUND 2 (informed by round 1 results)"); print("=" * 60)
        round2 = self.run_round(product_description, brief=round1["brief"], learnings=learnings)
        performance_r2 = generate_synthetic_performance(round2["content"], seed=99)
        self._log("performance_r2", str(performance_r2))

        return {
            "round1": round1, "round1_performance": performance,
            "round2": round2, "round2_performance": performance_r2,
            "learnings": learnings, "history": self.history,
        }


## 11. Run the Pipeline

Change `PRODUCT_DESCRIPTION` to anything you like — this is the only input the whole system needs.


In [ ]:
PRODUCT_DESCRIPTION = "A plant-based protein bar aimed at gym-goers"

orchestrator = MarketingTeamOrchestrator(max_revision_loops=2)
result = orchestrator.run_full_campaign(PRODUCT_DESCRIPTION, auto_approve_human=True)

print("\n\nPipeline finished. Result keys:", list(result.keys()))


## 12. Visualizations & Dashboard Components

Three charts: round-over-round CTR/conversion comparison, ROAS vs. a breakeven line, and how many revision loops each piece of content needed — a direct proxy for how well-calibrated the Writer and Editor's prompts are.


In [ ]:
import numpy as np

def plot_round_comparison(perf_r1: List[PerformanceRecord], perf_r2: List[PerformanceRecord]):
    channels = [p.channel for p in perf_r1]
    x = np.arange(len(channels))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    ctr1 = [p.ctr * 100 for p in perf_r1]; ctr2 = [p.ctr * 100 for p in perf_r2]
    axes[0].bar(x - width/2, ctr1, width, label="Round 1", color="#8C97AC")
    axes[0].bar(x + width/2, ctr2, width, label="Round 2", color="#4ADE9E")
    axes[0].set_xticks(x); axes[0].set_xticklabels(channels)
    axes[0].set_ylabel("CTR (%)"); axes[0].set_title("Click-Through Rate: Round 1 vs Round 2")
    axes[0].legend()

    conv1 = [p.conversion_rate * 100 for p in perf_r1]; conv2 = [p.conversion_rate * 100 for p in perf_r2]
    axes[1].bar(x - width/2, conv1, width, label="Round 1", color="#8C97AC")
    axes[1].bar(x + width/2, conv2, width, label="Round 2", color="#5AA9E6")
    axes[1].set_xticks(x); axes[1].set_xticklabels(channels)
    axes[1].set_ylabel("Conversion Rate (%)"); axes[1].set_title("Conversion Rate: Round 1 vs Round 2")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


def plot_roas(perf_r1: List[PerformanceRecord], perf_r2: List[PerformanceRecord]):
    channels = [p.channel for p in perf_r1]
    x = np.arange(len(channels)); width = 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - width/2, [p.roas for p in perf_r1], width, label="Round 1", color="#8C97AC")
    ax.bar(x + width/2, [p.roas for p in perf_r2], width, label="Round 2", color="#E8B84B")
    ax.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="Breakeven (1.0x)")
    ax.set_xticks(x); ax.set_xticklabels(channels)
    ax.set_ylabel("ROAS (x)"); ax.set_title("Return on Ad Spend by Channel")
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_revision_loops(round1_content: List[ContentPiece], round2_content: List[ContentPiece]):
    labels = [f"{c.channel} (R1)" for c in round1_content] + [f"{c.channel} (R2)" for c in round2_content]
    loops = [c.revision_loops for c in round1_content] + [c.revision_loops for c in round2_content]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(labels, loops, color="#5AA9E6")
    ax.set_xlabel("Revision loops needed before Editor approval")
    ax.set_title("Editor Revision Loops per Content Piece")
    plt.tight_layout()
    plt.show()


plot_round_comparison(result["round1_performance"], result["round2_performance"])
plot_roas(result["round1_performance"], result["round2_performance"])
plot_revision_loops(result["round1"]["content"], result["round2"]["content"])


## 13. Performance Metrics

Two different kinds of "performance" matter here, and it's worth being able to name the difference in an interview: **campaign performance** (did round 2's synthetic engagement improve?) and **pipeline performance** (is the system itself efficient — how many revision loops does it typically need, how many agent calls does a full run cost?).


In [ ]:
import pandas as pd

def pipeline_metrics_summary(result: dict) -> pd.DataFrame:
    r1_content = result["round1"]["content"]
    r2_content = result["round2"]["content"]
    all_content = r1_content + r2_content

    avg_loops = sum(c.revision_loops for c in all_content) / len(all_content)
    ctr1 = np.mean([p.ctr for p in result["round1_performance"]])
    ctr2 = np.mean([p.ctr for p in result["round2_performance"]])
    conv1 = np.mean([p.conversion_rate for p in result["round1_performance"]])
    conv2 = np.mean([p.conversion_rate for p in result["round2_performance"]])

    return pd.DataFrame([{
        "avg_revision_loops_per_piece": round(avg_loops, 2),
        "round1_avg_ctr": f"{ctr1:.2%}",
        "round2_avg_ctr": f"{ctr2:.2%}",
        "ctr_change": f"{(ctr2 - ctr1) / ctr1 * 100:+.1f}%",
        "round1_avg_conversion": f"{conv1:.2%}",
        "round2_avg_conversion": f"{conv2:.2%}",
        "conversion_change": f"{(conv2 - conv1) / conv1 * 100:+.1f}%",
        "total_logged_agent_steps": len(result["history"]),
    }])

metrics_df = pipeline_metrics_summary(result)
metrics_df.T.rename(columns={0: "value"})


## 14. Final Deliverables

Running this notebook end to end produces:

- A full campaign brief, generated from a single product description
- Two rounds of reviewed, on-brand marketing content across three channels
- A complete audit trail of every agent decision (research, strategy, every editor verdict, every revision loop, both rounds of performance data, and the analyst's recommendations)
- Three charts comparing round 1 vs round 2 performance and revision efficiency
- A one-row pipeline metrics summary, exportable as CSV


In [ ]:
# Export the full audit trail — this is what you'd hand to a marketing ops
# team as a record of exactly what the system did and why.
audit_df = pd.DataFrame(result["history"])
audit_df.to_csv("agent_audit_trail.csv", index=False)
metrics_df.to_csv("pipeline_metrics.csv", index=False)
print(f"Exported {len(audit_df)} audit-trail rows to agent_audit_trail.csv")
print("Exported pipeline_metrics.csv")
audit_df.tail(10)


## 15. Resume Description

> Designed and built a multi-agent marketing workflow (Python, Claude API) in which a Manager agent orchestrates Researcher, Strategist, Writer, Editor, and Analyst agents to plan, produce, and iteratively improve marketing campaigns — including a revision loop with structured-output parsing and retry/error-handling, a human-in-the-loop approval gate, and a synthetic-performance feedback loop that measurably informed a second content round.

## 16. Potential Upgrades

Roughly in order of how much they'd impress a technical interviewer for an agentic-marketing role:

1. **Real tool use for the Researcher** — wire in a live web-search API (Tavily, SerpAPI) instead of relying on the model's own training data, so research is genuinely current.
2. **Real performance data** — replace `generate_synthetic_performance` with a connector to the Meta Marketing API or Google Ads API, reading actual campaign results.
3. **Dynamic orchestration** — let the Manager itself be an LLM call that decides the next step from the current state, rather than the current mostly-fixed sequence with two hardcoded decision points (this is the difference between a "workflow" and a fully autonomous agent).
4. **A real UI** — wrap the human-approval step in a small Streamlit app so a non-technical reviewer can approve/reject without touching the notebook.
5. **Guardrails/content-moderation pass** — add an explicit safety/compliance check before the human checkpoint, separate from brand-voice editing.
6. **Persistent memory** — store past campaigns' briefs and outcomes in a vector database (e.g., Chroma) so the Strategist can retrieve "what worked last time for a similar product" instead of starting cold every run.
7. **Statistical rigor on the feedback loop** — instead of one synthetic performance sample per round, simulate enough impressions to run a real significance test on whether round 2 actually beat round 1, rather than eyeballing the bar chart.
